In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        os.path.join(dirname, filename)

print('Done')

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

Done


Epoch [1/5]: 100%|██████████| 157/157 [03:57<00:00,  1.51s/batch, loss=0.4559]
Epoch 1/5 | Loss: 0.4559
Epoch [2/5]: 100%|██████████| 157/157 [04:15<00:00,  1.63s/batch, loss=0.3742]
Epoch 2/5 | Loss: 0.3742
Epoch [3/5]: 100%|██████████| 157/157 [04:16<00:00,  1.64s/batch, loss=0.3188]
Epoch 3/5 | Loss: 0.3188
Epoch [4/5]: 100%|██████████| 157/157 [04:16<00:00,  1.63s/batch, loss=0.3132]
Epoch 4/5 | Loss: 0.3132
Epoch [5/5]: 100%|██████████| 157/157 [04:17<00:00,  1.64s/batch, loss=0.3022]
Epoch 5/5 | Loss: 0.3022
Generating Predictions: 100%|██████████| 2000/2000 [05:56<00:00,  5.61it/s]
Saved: /kaggle/working/submission_test_only.csv
     Image_ID  Label
0  Image_0001      0
1  Image_0002      7
2  Image_0003      5
3  Image_0004      1
4  Image_0005      4

In [2]:
# # ============================================================
# # DLP WEEK 9 - FLORA/FAUNA CLASSIFICATION
# # STRONG CUSTOM CNN - TRAIN FROM SCRATCH
# # NO PRETRAINED MODELS / NO PRETRAINED WEIGHTS
# # ============================================================

# import os
# import random
# import numpy as np
# import pandas as pd
# from PIL import Image
# from pathlib import Path

# import torch
# import torch.nn as nn
# import torch.nn.functional as F
# from torch.utils.data import Dataset, DataLoader
# from torchvision import transforms

# from sklearn.model_selection import train_test_split
# from sklearn.metrics import f1_score, classification_report

# # ============================================================
# # 1. CONFIG
# # ============================================================

# SEED = 42

# def seed_everything(seed=42):
#     random.seed(seed)
#     np.random.seed(seed)
#     torch.manual_seed(seed)
#     torch.cuda.manual_seed_all(seed)

# seed_everything(SEED)

# DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# DATA_ROOT = "/kaggle/input/competitions/dlp-26t2-week9-assignment"
# TRAIN_DIR = os.path.join(DATA_ROOT, "train")
# TEST_DIR = os.path.join(DATA_ROOT, "test")

# CLASS_NAMES = [
#     "Amphibia",
#     "Animalia",
#     "Arachnida",
#     "Aves",
#     "Fungi",
#     "Insecta",
#     "Mammalia",
#     "Mollusca",
#     "Plantae",
#     "Reptilia"
# ]

# NUM_CLASSES = len(CLASS_NAMES)

# IMG_SIZE = 256
# BATCH_SIZE = 96
# NUM_WORKERS = 4

# EPOCHS = 25

# LR = 3e-4
# WEIGHT_DECAY = 1e-4

# MODEL_PATH = "/kaggle/working/best_strong_scratch_model.pth"
# SUBMISSION_PATH = "/kaggle/working/submission.csv"

# print("=" * 70)
# print("DEVICE:", DEVICE)
# if torch.cuda.is_available():
#     print("GPU:", torch.cuda.get_device_name(0))
# print("=" * 70)


# # ============================================================
# # 2. COLLECT TRAIN IMAGES
# # ============================================================

# image_extensions = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

# samples = []

# for class_id, class_name in enumerate(CLASS_NAMES):

#     class_dir = os.path.join(TRAIN_DIR, class_name)

#     files = [
#         os.path.join(class_dir, f)
#         for f in os.listdir(class_dir)
#         if Path(f).suffix.lower() in image_extensions
#     ]

#     for f in files:
#         samples.append((f, class_id))

# print("\nClass distribution:")

# for class_id, class_name in enumerate(CLASS_NAMES):
#     count = sum(label == class_id for _, label in samples)
#     print(f"{class_id:2d} {class_name:12s}: {count}")

# print("\nTotal images:", len(samples))


# # ============================================================
# # 3. STRATIFIED TRAIN / VALIDATION SPLIT
# # ============================================================

# paths = np.array([x[0] for x in samples])
# labels = np.array([x[1] for x in samples])

# train_paths, val_paths, train_labels, val_labels = train_test_split(
#     paths,
#     labels,
#     test_size=0.15,
#     random_state=SEED,
#     stratify=labels
# )

# print("\nTrain images:", len(train_paths))
# print("Val images  :", len(val_paths))


# # ============================================================
# # 4. TRANSFORMS
# # ============================================================

# train_transform = transforms.Compose([
#     transforms.RandomResizedCrop(
#         IMG_SIZE,
#         scale=(0.70, 1.0),
#         ratio=(0.80, 1.25)
#     ),

#     transforms.RandomHorizontalFlip(p=0.5),

#     transforms.RandomRotation(15),

#     transforms.ColorJitter(
#         brightness=0.20,
#         contrast=0.20,
#         saturation=0.20,
#         hue=0.05
#     ),

#     transforms.ToTensor(),

#     transforms.Normalize(
#         mean=[0.485, 0.456, 0.406],
#         std=[0.229, 0.224, 0.225]
#     ),

#     transforms.RandomErasing(
#         p=0.15,
#         scale=(0.02, 0.12),
#         ratio=(0.3, 3.3)
#     )
# ])


# val_transform = transforms.Compose([
#     transforms.Resize((IMG_SIZE, IMG_SIZE)),

#     transforms.ToTensor(),

#     transforms.Normalize(
#         mean=[0.485, 0.456, 0.406],
#         std=[0.229, 0.224, 0.225]
#     )
# ])


# # ============================================================
# # 5. DATASET
# # ============================================================

# class FloraDataset(Dataset):

#     def __init__(self, paths, labels=None, transform=None):
#         self.paths = list(paths)
#         self.labels = None if labels is None else list(labels)
#         self.transform = transform

#     def __len__(self):
#         return len(self.paths)

#     def __getitem__(self, idx):

#         path = self.paths[idx]

#         try:
#             image = Image.open(path).convert("RGB")
#         except Exception:
#             # fallback for corrupt/problematic image
#             image = Image.new("RGB", (IMG_SIZE, IMG_SIZE))

#         if self.transform:
#             image = self.transform(image)

#         if self.labels is not None:
#             return image, self.labels[idx]

#         return image, os.path.basename(path)


# train_dataset = FloraDataset(
#     train_paths,
#     train_labels,
#     train_transform
# )

# val_dataset = FloraDataset(
#     val_paths,
#     val_labels,
#     val_transform
# )

# train_loader = DataLoader(
#     train_dataset,
#     batch_size=BATCH_SIZE,
#     shuffle=True,
#     num_workers=NUM_WORKERS,
#     pin_memory=True,
#     persistent_workers=True
# )

# val_loader = DataLoader(
#     val_dataset,
#     batch_size=BATCH_SIZE,
#     shuffle=False,
#     num_workers=NUM_WORKERS,
#     pin_memory=True,
#     persistent_workers=True
# )

# print("\nTrain batches:", len(train_loader))
# print("Val batches  :", len(val_loader))


# # ============================================================
# # 6. SQUEEZE-EXCITATION BLOCK
# # ============================================================

# class SEBlock(nn.Module):

#     def __init__(self, channels, reduction=8):
#         super().__init__()

#         hidden = max(channels // reduction, 8)

#         self.pool = nn.AdaptiveAvgPool2d(1)

#         self.fc = nn.Sequential(
#             nn.Conv2d(channels, hidden, 1),
#             nn.SiLU(inplace=True),
#             nn.Conv2d(hidden, channels, 1),
#             nn.Sigmoid()
#         )

#     def forward(self, x):

#         scale = self.pool(x)
#         scale = self.fc(scale)

#         return x * scale


# # ============================================================
# # 7. STRONG RESIDUAL BLOCK
# # ============================================================

# class ResidualBlock(nn.Module):

#     def __init__(
#         self,
#         in_channels,
#         out_channels,
#         stride=1,
#         drop=0.0
#     ):
#         super().__init__()

#         self.conv1 = nn.Conv2d(
#             in_channels,
#             out_channels,
#             kernel_size=3,
#             stride=stride,
#             padding=1,
#             bias=False
#         )

#         self.bn1 = nn.BatchNorm2d(out_channels)

#         self.conv2 = nn.Conv2d(
#             out_channels,
#             out_channels,
#             kernel_size=3,
#             stride=1,
#             padding=1,
#             bias=False
#         )

#         self.bn2 = nn.BatchNorm2d(out_channels)

#         self.se = SEBlock(out_channels)

#         self.dropout = nn.Dropout2d(drop) if drop > 0 else nn.Identity()

#         if stride != 1 or in_channels != out_channels:

#             self.shortcut = nn.Sequential(
#                 nn.Conv2d(
#                     in_channels,
#                     out_channels,
#                     kernel_size=1,
#                     stride=stride,
#                     bias=False
#                 ),
#                 nn.BatchNorm2d(out_channels)
#             )

#         else:
#             self.shortcut = nn.Identity()

#     def forward(self, x):

#         identity = self.shortcut(x)

#         out = self.conv1(x)
#         out = self.bn1(out)
#         out = F.silu(out, inplace=True)

#         out = self.conv2(out)
#         out = self.bn2(out)

#         out = self.se(out)

#         out = self.dropout(out)

#         out = out + identity

#         out = F.silu(out, inplace=True)

#         return out


# # ============================================================
# # 8. CUSTOM CNN - FROM SCRATCH
# # ============================================================

# class StrongFloraFaunaNet(nn.Module):

#     def __init__(self, num_classes=10):

#         super().__init__()

#         # ----------------------------------------------------
#         # Stem
#         # ----------------------------------------------------

#         self.stem = nn.Sequential(

#             nn.Conv2d(
#                 3,
#                 64,
#                 kernel_size=3,
#                 stride=2,
#                 padding=1,
#                 bias=False
#             ),

#             nn.BatchNorm2d(64),
#             nn.SiLU(inplace=True),

#             nn.Conv2d(
#                 64,
#                 64,
#                 kernel_size=3,
#                 stride=1,
#                 padding=1,
#                 bias=False
#             ),

#             nn.BatchNorm2d(64),
#             nn.SiLU(inplace=True)
#         )

#         # 128x128
#         self.stage1 = nn.Sequential(
#             ResidualBlock(64, 64, 1, 0.02),
#             ResidualBlock(64, 64, 1, 0.02),
#             ResidualBlock(64, 64, 1, 0.02)
#         )

#         # 64x64
#         self.stage2 = nn.Sequential(
#             ResidualBlock(64, 128, 2, 0.03),
#             ResidualBlock(128, 128, 1, 0.03),
#             ResidualBlock(128, 128, 1, 0.03)
#         )

#         # 32x32
#         self.stage3 = nn.Sequential(
#             ResidualBlock(128, 256, 2, 0.05),
#             ResidualBlock(256, 256, 1, 0.05),
#             ResidualBlock(256, 256, 1, 0.05),
#             ResidualBlock(256, 256, 1, 0.05)
#         )

#         # 16x16
#         self.stage4 = nn.Sequential(
#             ResidualBlock(256, 512, 2, 0.08),
#             ResidualBlock(512, 512, 1, 0.08),
#             ResidualBlock(512, 512, 1, 0.08)
#         )

#         # ----------------------------------------------------
#         # Classification head
#         # ----------------------------------------------------

#         self.pool = nn.AdaptiveAvgPool2d(1)

#         self.head = nn.Sequential(

#             nn.Flatten(),

#             nn.Dropout(0.30),

#             nn.Linear(512, 256),

#             nn.BatchNorm1d(256),

#             nn.SiLU(inplace=True),

#             nn.Dropout(0.20),

#             nn.Linear(256, num_classes)
#         )

#         self._initialize_weights()

#     def _initialize_weights(self):

#         for m in self.modules():

#             if isinstance(m, nn.Conv2d):

#                 nn.init.kaiming_normal_(
#                     m.weight,
#                     mode="fan_out",
#                     nonlinearity="relu"
#                 )

#             elif isinstance(m, nn.BatchNorm2d):

#                 nn.init.ones_(m.weight)
#                 nn.init.zeros_(m.bias)

#             elif isinstance(m, nn.BatchNorm1d):

#                 nn.init.ones_(m.weight)
#                 nn.init.zeros_(m.bias)

#             elif isinstance(m, nn.Linear):

#                 nn.init.trunc_normal_(
#                     m.weight,
#                     std=0.02
#                 )

#                 if m.bias is not None:
#                     nn.init.zeros_(m.bias)

#     def forward(self, x):

#         x = self.stem(x)

#         x = self.stage1(x)

#         x = self.stage2(x)

#         x = self.stage3(x)

#         x = self.stage4(x)

#         x = self.pool(x)

#         x = self.head(x)

#         return x


# # ============================================================
# # 9. CREATE MODEL
# # ============================================================

# model = StrongFloraFaunaNet(NUM_CLASSES).to(DEVICE)

# total_params = sum(
#     p.numel() for p in model.parameters()
# )

# trainable_params = sum(
#     p.numel()
#     for p in model.parameters()
#     if p.requires_grad
# )

# print("\n" + "=" * 70)
# print("MODEL")
# print("=" * 70)

# print(f"Total parameters     : {total_params / 1e6:.2f} M")
# print(f"Trainable parameters : {trainable_params / 1e6:.2f} M")

# print("\nIMPORTANT:")
# print("This model is trained COMPLETELY FROM SCRATCH.")
# print("NO pretrained weights are used.")


# # ============================================================
# # 10. LOSS + OPTIMIZER
# # ============================================================

# criterion = nn.CrossEntropyLoss(
#     label_smoothing=0.08
# )

# optimizer = torch.optim.AdamW(
#     model.parameters(),
#     lr=LR,
#     weight_decay=WEIGHT_DECAY
# )

# scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
#     optimizer,
#     T_max=EPOCHS,
#     eta_min=1e-6
# )

# scaler = torch.amp.GradScaler(
#     "cuda",
#     enabled=torch.cuda.is_available()
# )


# # ============================================================
# # 11. MIXUP
# # ============================================================

# def mixup_data(x, y, alpha=0.15):

#     if alpha <= 0:
#         return x, y, y, 1.0

#     lam = np.random.beta(alpha, alpha)

#     batch_size = x.size(0)

#     index = torch.randperm(
#         batch_size,
#         device=x.device
#     )

#     mixed_x = lam * x + (1 - lam) * x[index]

#     y_a = y
#     y_b = y[index]

#     return mixed_x, y_a, y_b, lam


# # ============================================================
# # 12. TRAIN ONE EPOCH
# # ============================================================

# def train_one_epoch(epoch):

#     model.train()

#     running_loss = 0.0

#     all_preds = []
#     all_targets = []

#     for batch_idx, (images, targets) in enumerate(train_loader):

#         images = images.to(
#             DEVICE,
#             non_blocking=True
#         )

#         targets = torch.as_tensor(
#             targets,
#             device=DEVICE,
#             dtype=torch.long
#         )

#         optimizer.zero_grad(set_to_none=True)

#         # MixUp only sometimes
#         use_mixup = (
#             epoch >= 3
#             and random.random() < 0.25
#         )

#         with torch.amp.autocast(
#             device_type="cuda",
#             enabled=torch.cuda.is_available()
#         ):

#             if use_mixup:

#                 images_mix, y_a, y_b, lam = mixup_data(
#                     images,
#                     targets,
#                     alpha=0.15
#                 )

#                 outputs = model(images_mix)

#                 loss = (
#                     lam * criterion(outputs, y_a)
#                     + (1 - lam) * criterion(outputs, y_b)
#                 )

#             else:

#                 outputs = model(images)

#                 loss = criterion(
#                     outputs,
#                     targets
#                 )

#         scaler.scale(loss).backward()

#         scaler.unscale_(optimizer)

#         torch.nn.utils.clip_grad_norm_(
#             model.parameters(),
#             max_norm=1.0
#         )

#         scaler.step(optimizer)
#         scaler.update()

#         running_loss += loss.item()

#         preds = outputs.argmax(dim=1)

#         all_preds.extend(
#             preds.detach().cpu().numpy()
#         )

#         all_targets.extend(
#             targets.detach().cpu().numpy()
#         )

#     train_f1 = f1_score(
#         all_targets,
#         all_preds,
#         average="micro"
#     )

#     return running_loss / len(train_loader), train_f1


# # ============================================================
# # 13. VALIDATION
# # ============================================================

# @torch.no_grad()
# def validate():

#     model.eval()

#     running_loss = 0.0

#     all_preds = []
#     all_targets = []

#     for images, targets in val_loader:

#         images = images.to(
#             DEVICE,
#             non_blocking=True
#         )

#         targets = torch.as_tensor(
#             targets,
#             device=DEVICE,
#             dtype=torch.long
#         )

#         with torch.amp.autocast(
#             device_type="cuda",
#             enabled=torch.cuda.is_available()
#         ):

#             outputs = model(images)

#             loss = criterion(
#                 outputs,
#                 targets
#             )

#         running_loss += loss.item()

#         preds = outputs.argmax(dim=1)

#         all_preds.extend(
#             preds.cpu().numpy()
#         )

#         all_targets.extend(
#             targets.cpu().numpy()
#         )

#     val_f1 = f1_score(
#         all_targets,
#         all_preds,
#         average="micro"
#     )

#     return (
#         running_loss / len(val_loader),
#         val_f1,
#         all_targets,
#         all_preds
#     )


# # ============================================================
# # 14. TRAINING
# # ============================================================

# print("\n" + "=" * 70)
# print("STARTING STRONG SCRATCH TRAINING")
# print("=" * 70)

# best_f1 = 0.0
# best_epoch = 0

# for epoch in range(1, EPOCHS + 1):

#     train_loss, train_f1 = train_one_epoch(epoch)

#     val_loss, val_f1, _, _ = validate()

#     scheduler.step()

#     current_lr = optimizer.param_groups[0]["lr"]

#     print(
#         f"Epoch {epoch:02d}/{EPOCHS} | "
#         f"Train Loss: {train_loss:.4f} | "
#         f"Train F1: {train_f1:.4f} | "
#         f"Val Loss: {val_loss:.4f} | "
#         f"Val F1: {val_f1:.5f} | "
#         f"LR: {current_lr:.7f}"
#     )

#     if val_f1 > best_f1:

#         best_f1 = val_f1
#         best_epoch = epoch

#         torch.save(
#             {
#                 "model_state_dict": model.state_dict(),
#                 "best_f1": best_f1,
#                 "epoch": epoch,
#                 "class_names": CLASS_NAMES
#             },
#             MODEL_PATH
#         )

#         print(
#             f"🔥 NEW BEST F1: {best_f1:.5f}"
#         )


# # ============================================================
# # 15. LOAD BEST MODEL
# # ============================================================

# checkpoint = torch.load(
#     MODEL_PATH,
#     map_location=DEVICE,
#     weights_only=False
# )

# model.load_state_dict(
#     checkpoint["model_state_dict"]
# )

# print("\n" + "=" * 70)
# print("BEST RESULT")
# print("=" * 70)

# print("Best Epoch :", checkpoint["epoch"])
# print("Best Val F1:", checkpoint["best_f1"])


# # ============================================================
# # 16. FINAL VALIDATION REPORT
# # ============================================================

# val_loss, val_f1, y_true, y_pred = validate()

# print("\nFinal Validation F1:", val_f1)

# print("\nClassification Report:\n")

# print(
#     classification_report(
#         y_true,
#         y_pred,
#         target_names=CLASS_NAMES,
#         digits=4
#     )
# )


# # ============================================================
# # 17. TEST DATA
# # ============================================================

# test_files = [
#     os.path.join(TEST_DIR, f)
#     for f in os.listdir(TEST_DIR)
#     if Path(f).suffix.lower() in image_extensions
# ]

# print("\nTest images:", len(test_files))


# # ============================================================
# # 18. TEST DATASET
# # ============================================================

# test_dataset = FloraDataset(
#     test_files,
#     labels=None,
#     transform=val_transform
# )

# test_loader = DataLoader(
#     test_dataset,
#     batch_size=BATCH_SIZE,
#     shuffle=False,
#     num_workers=NUM_WORKERS,
#     pin_memory=True,
#     persistent_workers=True
# )


# # ============================================================
# # 19. TEST PREDICTION
# # ============================================================

# @torch.no_grad()
# def predict_test():

#     model.eval()

#     predictions = []
#     image_ids = []

#     for images, filenames in test_loader:

#         images = images.to(
#             DEVICE,
#             non_blocking=True
#         )

#         with torch.amp.autocast(
#             device_type="cuda",
#             enabled=torch.cuda.is_available()
#         ):

#             outputs = model(images)

#         preds = outputs.argmax(dim=1)

#         predictions.extend(
#             preds.cpu().numpy()
#         )

#         image_ids.extend(
#             filenames
#         )

#     return image_ids, predictions


# image_ids, predictions = predict_test()


# # ============================================================
# # 20. CREATE SUBMISSION
# # ============================================================

# submission = pd.DataFrame({
#     "Image_ID": image_ids,
#     "Label": predictions
# })


# # ------------------------------------------------------------
# # Sort Image_ID numerically when possible
# # ------------------------------------------------------------

# def numeric_image_id(x):

#     try:
#         return int(
#             Path(str(x)).stem
#         )
#     except:
#         return str(x)


# submission["_sort"] = submission["Image_ID"].apply(
#     numeric_image_id
# )

# submission = submission.sort_values(
#     "_sort"
# ).drop(
#     columns=["_sort"]
# ).reset_index(drop=True)


# # ============================================================
# # 21. SAVE
# # ============================================================

# submission.to_csv(
#     SUBMISSION_PATH,
#     index=False
# )

# print("\n" + "=" * 70)
# print("SUBMISSION CREATED")
# print("=" * 70)

# print("File:", SUBMISSION_PATH)

# print("\nShape:", submission.shape)

# print("\nFirst 10 rows:")
# print(submission.head(10))

# print("\nLabel distribution:")
# print(
#     submission["Label"]
#     .value_counts()
#     .sort_index()
# )

# print("\nClass distribution:")

# for class_id, class_name in enumerate(CLASS_NAMES):

#     count = (
#         submission["Label"] == class_id
#     ).sum()

#     print(
#         f"{class_id} {class_name:12s}: {count}"
#     )

# print("\nDONE!")  12 epoch main 0.35 aaya tha 

In [3]:
# =====================================================================
# DLP 26T2 Week9 — WideResNet from Scratch (No Pretrained)
# 23F1002643 | T4 x2 | Target: 0.55-0.65 F1
# Fixes: WideResNet + CutMix + Class Weights + Single Cosine Cycle
# =====================================================================

import os, random, numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader, random_split, Subset
from torchvision import datasets, transforms
from torchvision.transforms import InterpolationMode
from PIL import Image
from tqdm.auto import tqdm
from torch.amp import GradScaler, autocast
from sklearn.metrics import f1_score, classification_report

# ── Config ────────────────────────────────────────────────────────────
SEED       = 42
IMG_SIZE   = 224
BATCH_SIZE = 96        # 45M model → thoda kam batch
EPOCHS     = 80        # Single long cosine cycle
VAL_RATIO  = 0.15
DATA_ROOT  = "/kaggle/input/competitions/dlp-26t2-week9-assignment"
TRAIN_DIR  = os.path.join(DATA_ROOT, "train")
TEST_DIR   = os.path.join(DATA_ROOT, "test")
OUTPUT_CSV = "/kaggle/working/submission.csv"
CKPT_PATH  = "/kaggle/working/best_model.pth"

# ── Seed ──────────────────────────────────────────────────────────────
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE} | GPUs: {torch.cuda.device_count()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# ── Transforms ────────────────────────────────────────────────────────
train_transform = transforms.Compose([
    transforms.Resize([256], interpolation=InterpolationMode.BICUBIC),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomRotation(degrees=30),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.6, 1.0)),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.1),
    transforms.RandomGrayscale(p=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.25, scale=(0.02, 0.2))
])

val_transform = transforms.Compose([
    transforms.Resize([256], interpolation=InterpolationMode.BICUBIC),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# ── Dataset Split ─────────────────────────────────────────────────────
base_dataset = datasets.ImageFolder(TRAIN_DIR)
class_names  = base_dataset.classes
NUM_CLASSES  = len(class_names)
print(f"Classes : {class_names}")
print(f"Total   : {len(base_dataset)} images")

generator  = torch.Generator().manual_seed(SEED)
train_size = int((1 - VAL_RATIO) * len(base_dataset))
val_size   = len(base_dataset) - train_size
train_idx, val_idx = random_split(
    range(len(base_dataset)), [train_size, val_size], generator=generator
)
train_idx = list(train_idx)
val_idx   = list(val_idx)

full_train    = datasets.ImageFolder(TRAIN_DIR, transform=train_transform)
full_val      = datasets.ImageFolder(TRAIN_DIR, transform=val_transform)
train_dataset = Subset(full_train, train_idx)
val_dataset   = Subset(full_val,   val_idx)
print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)}")

# ── Class Weights (imbalance fix) ─────────────────────────────────────
# Count per class in train split
train_labels = [base_dataset.targets[i] for i in train_idx]
class_counts = torch.zeros(NUM_CLASSES)
for lbl in train_labels:
    class_counts[lbl] += 1
class_weights = 1.0 / class_counts
class_weights = (class_weights / class_weights.sum() * NUM_CLASSES).to(DEVICE)
print(f"Class weights: {class_weights.cpu().numpy().round(3)}")

# ── Dataloaders ───────────────────────────────────────────────────────
num_cpus     = os.cpu_count()
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=num_cpus, pin_memory=True,
                          persistent_workers=True)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=num_cpus, pin_memory=True,
                          persistent_workers=True)
print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

# ── Model: WideResNet (45M params) ────────────────────────────────────
class BottleneckBlock(nn.Module):
    expansion = 4
    def __init__(self, in_ch, mid_ch, stride=1, groups=1, base_width=64):
        super().__init__()
        width  = int(mid_ch * (base_width / 64.)) * groups
        out_ch = mid_ch * self.expansion
        self.body = nn.Sequential(
            nn.Conv2d(in_ch, width, 1, bias=False),
            nn.BatchNorm2d(width), nn.ReLU(inplace=True),
            nn.Conv2d(width, width, 3, stride=stride,
                      padding=1, groups=groups, bias=False),
            nn.BatchNorm2d(width), nn.ReLU(inplace=True),
            nn.Conv2d(width, out_ch, 1, bias=False),
            nn.BatchNorm2d(out_ch)
        )
        self.shortcut = nn.Sequential()
        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_ch)
            )
    def forward(self, x):
        return F.relu(self.body(x) + self.shortcut(x))

class WideResNet(nn.Module):
    def __init__(self, num_classes=10, groups=2, base_width=64):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, 64, 7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.MaxPool2d(3, stride=2, padding=1)
        )
        self.layer1 = self._make_layer(64,   64,  3, 1, groups, base_width)
        self.layer2 = self._make_layer(256,  128, 4, 2, groups, base_width)
        self.layer3 = self._make_layer(512,  256, 6, 2, groups, base_width)
        self.layer4 = self._make_layer(1024, 512, 3, 2, groups, base_width)
        self.pool   = nn.AdaptiveAvgPool2d(1)
        self.head   = nn.Sequential(
            nn.Dropout(0.4),
            nn.Linear(2048, 512),
            nn.LayerNorm(512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )
        self._init_weights()

    def _make_layer(self, in_ch, mid_ch, blocks, stride, groups, base_width):
        layers = [BottleneckBlock(in_ch, mid_ch, stride, groups, base_width)]
        for _ in range(1, blocks):
            layers.append(BottleneckBlock(mid_ch*4, mid_ch, 1, groups, base_width))
        return nn.Sequential(*layers)

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight); nn.init.zeros_(m.bias)

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x); x = self.layer2(x)
        x = self.layer3(x); x = self.layer4(x)
        return self.head(self.pool(x).flatten(1))

model = WideResNet(num_classes=NUM_CLASSES, groups=2, base_width=64)
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs!")
    model = nn.DataParallel(model)
model = model.to(DEVICE)
params = sum(p.numel() for p in model.parameters())
print(f"Params: {params/1e6:.2f}M")

# ── Mixup ─────────────────────────────────────────────────────────────
def mixup(x, y, alpha=0.2):
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0), device=x.device)
    return lam*x + (1-lam)*x[idx], y, y[idx], lam

# ── CutMix ────────────────────────────────────────────────────────────
def cutmix(x, y, alpha=1.0):
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0), device=x.device)
    W, H = x.size(3), x.size(2)
    cut_ratio = np.sqrt(1 - lam)
    cut_w = int(W * cut_ratio)
    cut_h = int(H * cut_ratio)
    cx = np.random.randint(W)
    cy = np.random.randint(H)
    x1 = np.clip(cx - cut_w//2, 0, W)
    x2 = np.clip(cx + cut_w//2, 0, W)
    y1 = np.clip(cy - cut_h//2, 0, H)
    y2 = np.clip(cy + cut_h//2, 0, H)
    x_mix = x.clone()
    x_mix[:, :, y1:y2, x1:x2] = x[idx, :, y1:y2, x1:x2]
    lam = 1 - (x2-x1)*(y2-y1) / (W*H)
    return x_mix, y, y[idx], lam

# ── Loss / Optimizer / Scheduler ──────────────────────────────────────
criterion = nn.CrossEntropyLoss(
    weight=class_weights,
    label_smoothing=0.1
)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)

# Single long cosine — no restarts (restart pe F1 girta tha)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=EPOCHS, eta_min=1e-6
)
scaler  = GradScaler('cuda')
best_f1 = 0.0
torch.cuda.empty_cache()

# ── Validation ────────────────────────────────────────────────────────
@torch.no_grad()
def validate():
    model.eval()
    all_preds, all_labels = [], []
    total_loss = 0.0
    for images, labels in val_loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        with autocast('cuda'):
            outputs = model(images)
            loss    = criterion(outputs, labels)
        total_loss += loss.item() * images.size(0)
        all_preds.extend(outputs.argmax(dim=1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    f1       = f1_score(all_labels, all_preds, average='micro')
    avg_loss = total_loss / len(val_dataset)
    return avg_loss, f1, all_preds, all_labels

# ── Train Loop ────────────────────────────────────────────────────────
print("\n" + "="*70)
print("TRAINING START")
print("="*70)

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0

    with tqdm(train_loader, desc=f"Epoch {epoch+1:02d}/{EPOCHS}") as pbar:
        for images, labels in pbar:
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)

            # Augmentation strategy:
            # Epoch 0-4  : normal only (model stabilize ho)
            # Epoch 5+   : Mixup 40% | CutMix 40% | Normal 20%
            r = random.random()
            if epoch >= 5:
                if r < 0.4:
                    images, y_a, y_b, lam = mixup(images, labels, alpha=0.4)
                    aug = 'mixup'
                elif r < 0.8:
                    images, y_a, y_b, lam = cutmix(images, labels, alpha=1.0)
                    aug = 'cutmix'
                else:
                    aug = 'none'
            else:
                aug = 'none'

            optimizer.zero_grad(set_to_none=True)
            with autocast('cuda'):
                outputs = model(images)
                if aug != 'none':
                    loss = lam*criterion(outputs,y_a) + (1-lam)*criterion(outputs,y_b)
                else:
                    loss = criterion(outputs, labels)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer); scaler.update()

            total_loss += loss.item() * images.size(0)
            pbar.set_postfix({
                "loss": f"{loss.item():.4f}",
                "lr"  : f"{scheduler.get_last_lr()[0]:.6f}"
            })

    scheduler.step()

    train_loss = total_loss / len(train_dataset)
    val_loss, val_f1, val_preds, val_labels = validate()

    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save({
            "model_state_dict": model.state_dict(),
            "classes" : class_names,
            "val_f1"  : best_f1,
            "epoch"   : epoch + 1
        }, CKPT_PATH)
        print(f"  🔥 BEST F1: {best_f1:.5f} (saved)")

    print(f"Epoch {epoch+1:02d}/{EPOCHS} | "
          f"Train Loss: {train_loss:.4f} | "
          f"Val Loss: {val_loss:.4f} | "
          f"Val F1: {val_f1:.5f} | "
          f"LR: {scheduler.get_last_lr()[0]:.7f}")

# ── Load Best & Report ────────────────────────────────────────────────
print("\nLoading best model...")
ckpt = torch.load(CKPT_PATH, map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])
print(f"Best Epoch: {ckpt['epoch']} | Best Val F1: {ckpt['val_f1']:.5f}")

_, _, final_preds, final_labels = validate()
print("\nClassification Report:")
print(classification_report(final_labels, final_preds,
                             target_names=class_names, digits=4))

# ── TTA Inference ─────────────────────────────────────────────────────
tta_transforms = [
    val_transform,
    transforms.Compose([
        transforms.Resize([256], interpolation=InterpolationMode.BICUBIC),
        transforms.CenterCrop(IMG_SIZE),
        transforms.RandomHorizontalFlip(p=1.0),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
    ]),
    transforms.Compose([
        transforms.Resize([256], interpolation=InterpolationMode.BICUBIC),
        transforms.CenterCrop(IMG_SIZE),
        transforms.RandomVerticalFlip(p=1.0),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
    ]),
    transforms.Compose([
        transforms.Resize([280], interpolation=InterpolationMode.BICUBIC),
        transforms.CenterCrop(IMG_SIZE),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
    ]),
    transforms.Compose([
        transforms.Resize([256], interpolation=InterpolationMode.BICUBIC),
        transforms.CenterCrop(IMG_SIZE),
        transforms.RandomRotation(degrees=(90,90)),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
    ]),
]

model.eval()
image_files = sorted([f for f in os.listdir(TEST_DIR)
                       if f.lower().endswith(('.jpg','.jpeg','.png'))])
results = []

print(f"\nTTA Inference on {len(image_files)} images...")
for fname in tqdm(image_files, desc="TTA"):
    img   = Image.open(os.path.join(TEST_DIR, fname)).convert('RGB')
    probs = torch.zeros(NUM_CLASSES).to(DEVICE)
    for tfm in tta_transforms:
        with torch.no_grad():
            out    = model(tfm(img).unsqueeze(0).to(DEVICE))
            probs += torch.softmax(out, dim=1).squeeze()
    results.append({"Image_ID": fname.split('.')[0],
                    "Label":    probs.argmax().item()})

df = pd.DataFrame(results)
df['sort_key'] = df['Image_ID'].str.extract(r'(\d+)').astype(int)
df = df.sort_values('sort_key').drop('sort_key', axis=1).reset_index(drop=True)
df.to_csv(OUTPUT_CSV, index=False)

print(f"\n✅ Saved → {OUTPUT_CSV}")
print(f"Total: {len(df)}")
print("\nLabel distribution:")
print(df['Label'].value_counts().sort_index())
print("\nSample:")
print(df.head(10))

Device: cuda | GPUs: 2
GPU: Tesla T4
Classes : ['Amphibia', 'Animalia', 'Arachnida', 'Aves', 'Fungi', 'Insecta', 'Mammalia', 'Mollusca', 'Plantae', 'Reptilia']
Total   : 9999 images
Train: 8499 | Val: 1500
Class weights: [1.013 1.007 0.996 1.025 0.99  0.992 0.977 1.004 1.013 0.983]
Train batches: 89 | Val batches: 16
Using 2 GPUs!
Params: 45.25M

TRAINING START


Epoch 01/80:   0%|          | 0/89 [00:00<?, ?it/s]

  🔥 BEST F1: 0.20800 (saved)
Epoch 01/80 | Train Loss: 2.4442 | Val Loss: 2.2167 | Val F1: 0.20800 | LR: 0.0002999


Epoch 02/80:   0%|          | 0/89 [00:00<?, ?it/s]

  🔥 BEST F1: 0.22333 (saved)
Epoch 02/80 | Train Loss: 2.2747 | Val Loss: 2.1746 | Val F1: 0.22333 | LR: 0.0002995


...

Epoch 79/80 | Train Loss: 1.8934 | Val Loss: 1.6583 | Val F1: 0.47467 | LR: 0.0000011


Epoch 80/80:   0%|          | 0/89 [00:00<?, ?it/s]

Epoch 80/80 | Train Loss: 1.8741 | Val Loss: 1.6589 | Val F1: 0.47067 | LR: 0.0000010

Loading best model...
Best Epoch: 74 | Best Val F1: 0.48067

Classification Report:
              precision    recall  f1-score   support

    Amphibia     0.3945    0.2671    0.3185       161
    Animalia     0.5000    0.5321    0.5155       156
   Arachnida     0.7477    0.5646    0.6434       147
        Aves     0.6024    0.5848    0.5935       171
       Fungi     0.5556    0.4610    0.5039       141
     Insecta     0.4362    0.4545    0.4452       143
    Mammalia     0.4286    0.5538    0.4832       130
    Mollusca     0.5309    0.2792    0.3660       154
     Plantae     0.5045    0.6957    0.5849       161
    Reptilia     0.2607    0.4044    0.3170       136

    accuracy                         0.4807      1500
   macro avg     0.4961    0.4797    0.4771      1500
weighted avg     0.4995    0.4807    0.4793      1500


TTA Inference on 2000 images...


TTA:   0%|          | 0/2000 [00:00<?, ?it/s]


✅ Saved → /kaggle/working/submission.csv
Total: 2000

Label distribution:
Label
0    123
1    212
2    170
3    193
4    173
5    201
6    221
7    102
8    298
9    307
Name: count, dtype: int64

Sample:
     Image_ID  Label
0  Image_0001      0
1  Image_0002      4
2  Image_0003      5
3  Image_0004      6
4  Image_0005      9
5  Image_0006      5
6  Image_0007      6
7  Image_0008      0
8  Image_0009      0
9  Image_0010      9
